In [12]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
import numpy as np

In [13]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:106.0) Gecko/20100101 Firefox/106.0',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,/;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    # 'Accept-Encoding': 'gzip, deflate, br',
    'DNT': '1',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
    'Sec-Fetch-Dest': 'document',
    'Sec-Fetch-Mode': 'navigate',
    'Sec-Fetch-Site': 'none',
    'Sec-Fetch-User': '?1',
}

In [3]:
url = "https://collegedunia.com"

In [4]:
page = requests.get(url, headers = headers)

In [5]:
page

<Response [200]>

In [86]:
ranks1 = []
out_of_ranks1 = []
years1 = []
avg_packages1 = []
high_packages1 = []

for page_no in range(5):
    url=f'https://collegedunia.com/btech-colleges?page={page_no}'
    page=requests.get(url,headers=headers)
    soup=BeautifulSoup(page.text)

    r=soup.find_all('span',class_='jsx-2794970405 rank-container pointer d-block font-weight-medium text-dark-gray text-lg')
    for i in r:
        ranks1.append(re.findall(r'#(\d+)\w+/\d+\s\w+\s\w+\s\w+\s\w+\s\d+',i.text))
        out_of_ranks1.append(re.findall(r'#\d+\w+/(\d+)\s\w+\s\w+\s\w+\s\w+\s\d+',i.text))
        years1.append(re.findall(r'#\d+\w+/\d+\s\w+\s\w+\s\w+\s\w+\s(\d+)',i.text))
        
    avg=soup.find_all('span',class_='jsx-914129990 text-green d-block mb-1')
    for i in avg:
        avg_packages1.append(i.text)
    hig=soup.find_all('span',class_='jsx-914129990 text-green d-block mb-1')
    for i in hig:
        high_packages1.append(i.text)

In [6]:
courses = []
names = []
cities = []
states = []
fees = []
ratings = []
ranks = []
out_of_ranks = []
years = []
avg_packages = []
high_packages = []
approves = []

for page_no in range(40):  
    url = f"https://collegedunia.com/btech-colleges?page={page_no}"
    page = requests.get(url, headers=headers)
    soup = BeautifulSoup(page.text)

    

    co = soup.find_all("span", class_="jsx-3698117056 fee-shorm-form")
    for i in co:
        courses.append(i.text.strip())

        
        
    na = soup.find_all("h3", class_="jsx-3698117056 font-weight-medium text-lg mb-0")
    for i in na:
        name = i.text.strip()
        name = re.sub(r",.*|\[.*?\]|-", "", name).strip()
        names.append(name)

        

    ci = soup.find_all("span",class_ = "jsx-3698117056 pr-1 location")
    for i in ci:
        city = i.text.split(",")[0]
        cities.append(city)

        

    for i in ci:
        state = i.text.split(",")[1]
        states.append(state)


        

    pr = soup.find_all("span",class_ = "jsx-3698117056 text-lg text-green d-block font-weight-bold mb-1")
    for i in pr:
        price = i.text
        if price == "--":
            fees.append(np.nan)
        else:
            fees.append(price)

    


    
    rows = soup.find_all("tr", class_="table-row") 
    
    for row in rows:
        review_col = row.find("td", class_="col-reviews")
        
        if review_col:
            valid_rating = review_col.find("span", class_="lr-key")
            
            if valid_rating:
                raw_text = valid_rating.text.strip()
                clean_rating = raw_text.split('/')[0].strip()
                ratings.append(clean_rating)
            else:
                ratings.append(np.nan)
                
        else:
            ratings.append(np.nan)





    
    for row in rows:
        tag = row.find("span", class_="jsx-2794970405")
        
        if tag:
            text = tag.text.strip()
            
            if "#" in text and "/" in text:
                ranks.append(text.split("#")[1].split("th")[0].split("/")[0])
                out_of_ranks.append(text.split("/")[1].split()[0])
                
                last_word = text.split()[-1]
                if last_word.isdigit() and len(last_word) == 4:
                    years.append(last_word)
                else:
                    years.append(np.nan)
                    
            else:
                ranks.append(np.nan)
                out_of_ranks.append(np.nan)
                years.append(np.nan)
                
        else:
            ranks.append(np.nan)
            out_of_ranks.append(np.nan)
            years.append(np.nan)



    

    for row in rows:
        col = row.find("td", class_="col-placement")
        
        if col:
            text = col.text.strip()
            
            if "Average Package" in text:
                # 1. Split to get the left side
                raw_avg = text.split("Average Package")[0]
                
                # 2. Remove '₹' and use .strip() to remove the '\xa0' and extra spaces
                clean_avg = raw_avg.replace('₹', '').strip()
                
                avg_packages.append(clean_avg)
            else:
                avg_packages.append(np.nan)
                
        else:
            avg_packages.append(np.nan)




    
    for row in rows:
        col = row.find("td", class_="col-placement")
        
        if col:
            text = col.text.strip()
            
            if "Highest Package" in text:
                parts = text.split("Highest Package")[0].strip()
                raw_high = parts.split("₹")[-1]
                clean_high = raw_high.replace('₹', '').strip()
                high_packages.append(clean_high)
            else:
                high_packages.append(np.nan)
                
        else:
            high_packages.append(np.nan)




    app=soup.find_all('div',class_='jsx-3698117056 mt-1 font-weight-medium text-sm')
    for i in app:
        # print(i.text)
        if 'Approved' in i.text:
            approves.append(i.text.split('|')[1].rstrip('Approved'))
        else:
            approves.append(np.nan)
    

In [7]:
len(names)

420

In [8]:
df_btech = pd.DataFrame({"College Names":names,
                          "Degress":"BE/B.Tech",
                          "Course":courses,
                          "City":cities,
                          "State":states,
                          "Rating for 5":ratings,
                          "Rank":ranks,
                          "Total Rank":out_of_ranks,
                          "Year of Rank":years,
                          "Fees in Ruppes":fees,
                          "Average Package":avg_packages,
                          "Highest Package":high_packages,
                          "Approved":approves})

In [9]:
df_btech

,College Names,Degress,Course,City,State,Rating for 5,Rank,Total Rank,Year of Rank,Fees in Ruppes,Average Package,Highest Package,Approved
0,IIT Bombay Indian Institute of Technology,BE/B.Tech,B.Tech Computer Science and Engineering,Mumbai,Maharashtra,4.4,1,500,2025,"₹ 8,82,500","23,50,000","1,00,00,000","AICTE, UGC"
1,IIT Delhi Indian Institute of Technology,BE/B.Tech,B.Tech Computer Science and Engineering,New Delhi,Delhi NCR,4.3,2,500,2025,"₹ 8,62,550","25,82,000","2,00,00,000",NaN
2,IIT Madras Indian Institute of Technology,BE/B.Tech,B.Tech Computer Science and Engineering,Chennai,Tamil Nadu,4.3,3,500,2025,"₹ 9,38,668","21,48,000","4,30,00,000",AICTE
3,IIT Kanpur Indian Institute of Technology,BE/B.Tech,B.Tech Computer Science and Engineering,Kanpur,Uttar Pradesh,4.4,4,500,2025,"₹ 9,18,880","17,20,000","2,50,00,000",UGC
4,IIT Kharagpur Indian Institute of Technology,BE/B.Tech,B.Tech Mechanical Engineering,Kharagpur,West Bengal,4.4,5,500,2025,"₹ 8,96,300","24,30,000","2,44,00,000","AICTE, UGC, NBA"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
415,Nitra Technical Campus,BE/B.Tech,B.Tech Computer Science and Engineering,Ghaziabad,Uttar Pradesh,3.6,436,500,2025,"₹ 3,62,460","4,00,000","12,00,000",AICTE
416,ABES Institute of Technology,BE/B.Tech,B.Tech Computer Science and Engineering,Ghaziabad,Uttar Pradesh,3.9,437,500,2025,"₹ 5,57,400","5,50,000","27,00,000","AICTE, NBA"
417,Institute of Technology & Management,BE/B.Tech,B.Tech Computer Science and Engineering,Lucknow,Uttar Pradesh,3.6,438,500,2025,"₹ 2,66,600",NaN,"8,60,000",AICTE
418,Amity University,BE/B.Tech,B.Tech Computer Science and Engineering,Lucknow,Uttar Pradesh,4.1,439,500,2025,"₹ 15,36,000","8,00,000","45,00,000","NCTE, RCI, COA, PCI, BCI"


In [10]:
df_btech.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 420 entries, 0 to 419
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   College Names    420 non-null    object
 1   Degress          420 non-null    object
 2   Course           420 non-null    object
 3   City             420 non-null    object
 4   State            420 non-null    object
 5   Rating for 5     417 non-null    object
 6   Rank             418 non-null    object
 7   Total Rank       418 non-null    object
 8   Year of Rank     418 non-null    object
 9   Fees in Ruppes   417 non-null    object
 10  Average Package  303 non-null    object
 11  Highest Package  369 non-null    object
 12  Approved         406 non-null    object
dtypes: object(13)
memory usage: 42.8+ KB


In [21]:
courses = []
names = []
cities = []
states = []
fees = []
ratings = []
ranks = []
out_of_ranks = []
years = []
avg_packages = []
high_packages = []
approves = []

for page_no in range(30):  
    url = f"https://collegedunia.com/mtech-colleges?page={page_no}"
    page = requests.get(url, headers=headers)
    soup = BeautifulSoup(page.text)

    

    co = soup.find_all("span", class_="jsx-3698117056 fee-shorm-form")
    for i in co:
        courses.append(i.text.strip())

        
        
    na = soup.find_all("h3", class_="jsx-3698117056 font-weight-medium text-lg mb-0")
    for i in na:
        name = i.text.strip()
        name = re.sub(r",.*|\[.*?\]|-", "", name).strip()
        names.append(name)

        

    ci = soup.find_all("span",class_ = "jsx-3698117056 pr-1 location")
    for i in ci:
        city = i.text.split(",")[0]
        cities.append(city)

        

    for i in ci:
        state = i.text.split(",")[1]
        states.append(state)


        

    pr = soup.find_all("span",class_ = "jsx-3698117056 text-lg text-green d-block font-weight-bold mb-1")
    for i in pr:
        price = i.text
        if price == "--":
            fees.append(np.nan)
        else:
            fees.append(price)

    


    
    rows = soup.find_all("tr", class_="table-row") 
    
    for row in rows:
        review_col = row.find("td", class_="col-reviews")
        
        if review_col:
            valid_rating = review_col.find("span", class_="lr-key")
            
            if valid_rating:
                raw_text = valid_rating.text.strip()
                clean_rating = raw_text.split('/')[0].strip()
                ratings.append(clean_rating)
            else:
                ratings.append(np.nan)
                
        else:
            ratings.append(np.nan)





    
    for row in rows:
        tag = row.find("span", class_="jsx-2794970405")
        
        if tag:
            text = tag.text.strip()
            
            if "#" in text and "/" in text:
                ranks.append(text.split("#")[1].split("th")[0].split("/")[0])
                out_of_ranks.append(text.split("/")[1].split()[0])
                
                last_word = text.split()[-1]
                if last_word.isdigit() and len(last_word) == 4:
                    years.append(last_word)
                else:
                    years.append(np.nan)
                    
            else:
                ranks.append(np.nan)
                out_of_ranks.append(np.nan)
                years.append(np.nan)
                
        else:
            ranks.append(np.nan)
            out_of_ranks.append(np.nan)
            years.append(np.nan)



    

    for row in rows:
        col = row.find("td", class_="col-placement")
        
        if col:
            text = col.text.strip()
            
            if "Average Package" in text:
                # 1. Split to get the left side
                raw_avg = text.split("Average Package")[0]
                
                # 2. Remove '₹' and use .strip() to remove the '\xa0' and extra spaces
                clean_avg = raw_avg.replace('₹', '').strip()
                
                avg_packages.append(clean_avg)
            else:
                avg_packages.append(np.nan)
                
        else:
            avg_packages.append(np.nan)




    
    for row in rows:
        col = row.find("td", class_="col-placement")
        
        if col:
            text = col.text.strip()
            
            if "Highest Package" in text:
                parts = text.split("Highest Package")[0].strip()
                raw_high = parts.split("₹")[-1]
                clean_high = raw_high.replace('₹', '').strip()
                high_packages.append(clean_high)
            else:
                high_packages.append(np.nan)
                
        else:
            high_packages.append(np.nan)




    app=soup.find_all('div',class_='jsx-3698117056 mt-1 font-weight-medium text-sm')
    for i in app:
        # print(i.text)
        if 'Approved' in i.text:
            approves.append(i.text.split('|')[1].rstrip('Approved'))
        else:
            approves.append(np.nan)
    

In [22]:
len(names)

320

In [23]:
df_mtech = pd.DataFrame({"College Names":names,
                          "Degress":"M.Tech",
                          "Course":courses,
                          "City":cities,
                          "State":states,
                          "Rating for 5":ratings,
                          "Rank":ranks,
                          "Total Rank":out_of_ranks,
                          "Year of Rank":years,
                          "Fees in Ruppes":fees,
                          "Average Package":avg_packages,
                          "Highest Package":high_packages,
                          "Approved":approves})

In [24]:
df_mtech

,College Names,Degress,Course,City,State,Rating for 5,Rank,Total Rank,Year of Rank,Fees in Ruppes,Average Package,Highest Package,Approved
0,IIT Bombay Indian Institute of Technology,M.Tech,M.Tech Computer Science And Engineering,Mumbai,Maharashtra,4.4,1,500,2025,"₹ 1,23,500","23,50,000","1,00,00,000","AICTE, UGC"
1,IIT Delhi Indian Institute of Technology,M.Tech,M.Tech Computer Science And Engineering,New Delhi,Delhi NCR,4.3,2,500,2025,"₹ 3,80,600","25,82,000","2,00,00,000",NaN
2,IIT Madras Indian Institute of Technology,M.Tech,M.Tech Web-Enabled,Chennai,Tamil Nadu,4.3,3,500,2025,"₹ 1,81,834","21,48,000","4,30,00,000",AICTE
3,IIT Kanpur Indian Institute of Technology,M.Tech,M.Tech Computer Science And Engineering,Kanpur,Uttar Pradesh,4.4,4,500,2025,"₹ 84,040","17,20,000","2,50,00,000",UGC
4,IIT Kharagpur Indian Institute of Technology,M.Tech,M.Tech Computer Science And Data Processing,Kharagpur,West Bengal,4.4,5,500,2025,"₹ 63,800","24,30,000","2,44,00,000","AICTE, UGC, NBA"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,Manakula Vinayagar Institute of Technology,M.Tech,M.Tech Computer Science And Engineering,Pondicherry,Puducherry,3.7,373,500,2025,"₹ 90,000","3,00,000","7,00,000","AICTE, NBA"
316,Mangalore Institute of Technology and Engineering,M.Tech,M.Tech Computer Science And Engineering,Mangalore,Karnataka,4.1,375,500,2025,"₹ 1,50,910",NaN,"50,00,000",AICTE
317,Laxminarayan Institute of Technology,M.Tech,M.Tech Chemical Engineering,Nagpur,Maharashtra,3.8,377,500,2025,"₹ 3,00,000","3,50,000","8,00,000","AICTE, UGC, NBA"
318,Gurukul Vidyapeeth,M.Tech,M.Tech Mechanical Engineering,Chandigarh,Chandigarh,1.8,NaN,NaN,NaN,"₹ 1,59,800",NaN,NaN,AICTE


In [25]:
df_mtech.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   College Names    320 non-null    object
 1   Degress          320 non-null    object
 2   Course           320 non-null    object
 3   City             320 non-null    object
 4   State            320 non-null    object
 5   Rating for 5     319 non-null    object
 6   Rank             319 non-null    object
 7   Total Rank       319 non-null    object
 8   Year of Rank     319 non-null    object
 9   Fees in Ruppes   315 non-null    object
 10  Average Package  254 non-null    object
 11  Highest Package  292 non-null    object
 12  Approved         312 non-null    object
dtypes: object(13)
memory usage: 32.6+ KB


In [26]:
courses = []
names = []
cities = []
states = []
fees = []
ratings = []
ranks = []
out_of_ranks = []
years = []
avg_packages = []
high_packages = []
approves = []

for page_no in range(30):  
    url = f"https://collegedunia.com/top-mba-colleges-in-india?page={page_no}"
    page = requests.get(url, headers=headers)
    soup = BeautifulSoup(page.text)

    

    co = soup.find_all("span", class_="jsx-3698117056 fee-shorm-form")
    for i in co:
        courses.append(i.text.strip())

        
        
    na = soup.find_all("h3", class_="jsx-3698117056 font-weight-medium text-lg mb-0")
    for i in na:
        name = i.text.strip()
        name = re.sub(r",.*|\[.*?\]|-", "", name).strip()
        names.append(name)

        

    ci = soup.find_all("span",class_ = "jsx-3698117056 pr-1 location")
    for i in ci:
        city = i.text.split(",")[0]
        cities.append(city)

        

    for i in ci:
        state = i.text.split(",")[1]
        states.append(state)


        

    pr = soup.find_all("span",class_ = "jsx-3698117056 text-lg text-green d-block font-weight-bold mb-1")
    for i in pr:
        price = i.text
        if price == "--":
            fees.append(np.nan)
        else:
            fees.append(price)

    


    
    rows = soup.find_all("tr", class_="table-row") 
    
    for row in rows:
        review_col = row.find("td", class_="col-reviews")
        
        if review_col:
            valid_rating = review_col.find("span", class_="lr-key")
            
            if valid_rating:
                raw_text = valid_rating.text.strip()
                clean_rating = raw_text.split('/')[0].strip()
                ratings.append(clean_rating)
            else:
                ratings.append(np.nan)
                
        else:
            ratings.append(np.nan)





    
    for row in rows:
        tag = row.find("span", class_="jsx-2794970405")
        
        if tag:
            text = tag.text.strip()
            
            if "#" in text and "/" in text:
                ranks.append(text.split("#")[1].split("th")[0].split("/")[0])
                out_of_ranks.append(text.split("/")[1].split()[0])
                
                last_word = text.split()[-1]
                if last_word.isdigit() and len(last_word) == 4:
                    years.append(last_word)
                else:
                    years.append(np.nan)
                    
            else:
                ranks.append(np.nan)
                out_of_ranks.append(np.nan)
                years.append(np.nan)
                
        else:
            ranks.append(np.nan)
            out_of_ranks.append(np.nan)
            years.append(np.nan)



    

    for row in rows:
        col = row.find("td", class_="col-placement")
        
        if col:
            text = col.text.strip()
            
            if "Average Package" in text:
                # 1. Split to get the left side
                raw_avg = text.split("Average Package")[0]
                
                # 2. Remove '₹' and use .strip() to remove the '\xa0' and extra spaces
                clean_avg = raw_avg.replace('₹', '').strip()
                
                avg_packages.append(clean_avg)
            else:
                avg_packages.append(np.nan)
                
        else:
            avg_packages.append(np.nan)




    
    for row in rows:
        col = row.find("td", class_="col-placement")
        
        if col:
            text = col.text.strip()
            
            if "Highest Package" in text:
                parts = text.split("Highest Package")[0].strip()
                raw_high = parts.split("₹")[-1]
                clean_high = raw_high.replace('₹', '').strip()
                high_packages.append(clean_high)
            else:
                high_packages.append(np.nan)
                
        else:
            high_packages.append(np.nan)




    app=soup.find_all('div',class_='jsx-3698117056 mt-1 font-weight-medium text-sm')
    for i in app:
        # print(i.text)
        if 'Approved' in i.text:
            approves.append(i.text.split('|')[1].rstrip('Approved'))
        else:
            approves.append(np.nan)
    

In [27]:
len(names)

320

In [28]:
df_mba = pd.DataFrame({"College Names":names,
                      "Degress":"MBA",
                      "Course":courses,
                      "City":cities,
                      "State":states,
                      "Rating for 5":ratings,
                      "Rank":ranks,
                      "Total Rank":out_of_ranks,
                      "Year of Rank":years,
                      "Fees in Ruppes":fees,
                      "Average Package":avg_packages,
                      "Highest Package":high_packages,
                      "Approved":approves})

In [29]:
df_mba

,College Names,Degress,Course,City,State,Rating for 5,Rank,Total Rank,Year of Rank,Fees in Ruppes,Average Package,Highest Package,Approved
0,IIMA Indian Institute of Management,MBA,PG Program Food And Agri-Business Management,Ahmedabad,Gujarat,4.6,1,372,2025,"₹ 27,50,000","35,22,591","1,10,00,000",UGC
1,IIMC Indian Institute of Management,MBA,MBA,Kolkata,West Bengal,4.4,2,372,2025,"₹ 27,00,000","34,23,000","1,45,00,000","UGC, AACSB, AMBA"
2,IIM Bangalore Indian Institute of Management,MBA,PGP Business Analytics,Bangalore,Karnataka,4.6,3,372,2025,"₹ 26,30,000","34,88,000","1,15,00,000","AICTE, UGC"
3,ISB Hyderabad Indian School of Business,MBA,PGPM,Hyderabad,Telangana,4.6,4,372,2025,"₹ 45,12,160","33,25,000","66,00,000","AACSB, AMBA"
4,Indian School of Business,MBA,PGPM,Mohali,Punjab,4.6,5,372,2025,"₹ 45,12,160","33,25,000","66,00,000","AACSB, AMBA"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,Institute of Engineering and Technology,MBA,MBA,Lucknow,Uttar Pradesh,3.8,NaN,NaN,NaN,"₹ 1,54,550","8,20,000","49,00,000","AICTE, UGC"
316,JNTUH,MBA,MBA Marketing,Hyderabad,Telangana,4,62,100,2025,"₹ 2,00,000","3,30,000","6,60,000","AICTE, UGC, NBA, MHRD"
317,Institute of Management Studies,MBA,MBA,Indore,Madhya Pradesh,3.9,88,100,2025,"₹ 1,98,000","6,15,998","18,50,000",NaN
318,NIST University,MBA,MBA Production & Operations,Berhampur,Odisha,3.9,NaN,NaN,NaN,"₹ 6,31,000",NaN,"23,00,000","AICTE, UGC, MHRD"


In [30]:
df_mba.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   College Names    320 non-null    object
 1   Degress          320 non-null    object
 2   Course           320 non-null    object
 3   City             320 non-null    object
 4   State            320 non-null    object
 5   Rating for 5     311 non-null    object
 6   Rank             314 non-null    object
 7   Total Rank       314 non-null    object
 8   Year of Rank     314 non-null    object
 9   Fees in Ruppes   318 non-null    object
 10  Average Package  264 non-null    object
 11  Highest Package  291 non-null    object
 12  Approved         304 non-null    object
dtypes: object(13)
memory usage: 32.6+ KB


In [31]:
courses = []
names = []
cities = []
states = []
fees = []
ratings = []
ranks = []
out_of_ranks = []
years = []
avg_packages = []
high_packages = []
approves = []

for page_no in range(29):  
    url = f"https://collegedunia.com/bcom-colleges?page={page_no}"
    page = requests.get(url, headers=headers)
    soup = BeautifulSoup(page.text)

    

    co = soup.find_all("span", class_="jsx-3698117056 fee-shorm-form")
    for i in co:
        courses.append(i.text.strip())

        
        
    na = soup.find_all("h3", class_="jsx-3698117056 font-weight-medium text-lg mb-0")
    for i in na:
        name = i.text.strip()
        name = re.sub(r",.*|\[.*?\]|-", "", name).strip()
        names.append(name)

        

    ci = soup.find_all("span",class_ = "jsx-3698117056 pr-1 location")
    for i in ci:
        city = i.text.split(",")[0]
        cities.append(city)

        

    for i in ci:
        state = i.text.split(",")[1]
        states.append(state)


        

    pr = soup.find_all("span",class_ = "jsx-3698117056 text-lg text-green d-block font-weight-bold mb-1")
    for i in pr:
        price = i.text
        if price == "--":
            fees.append(np.nan)
        else:
            fees.append(price)

    


    
    rows = soup.find_all("tr", class_="table-row") 
    
    for row in rows:
        review_col = row.find("td", class_="col-reviews")
        
        if review_col:
            valid_rating = review_col.find("span", class_="lr-key")
            
            if valid_rating:
                raw_text = valid_rating.text.strip()
                clean_rating = raw_text.split('/')[0].strip()
                ratings.append(clean_rating)
            else:
                ratings.append(np.nan)
                
        else:
            ratings.append(np.nan)





    
    for row in rows:
        tag = row.find("span", class_="jsx-2794970405")
        
        if tag:
            text = tag.text.strip()
            
            if "#" in text and "/" in text:
                ranks.append(text.split("#")[1].split("th")[0].split("/")[0])
                out_of_ranks.append(text.split("/")[1].split()[0])
                
                last_word = text.split()[-1]
                if last_word.isdigit() and len(last_word) == 4:
                    years.append(last_word)
                else:
                    years.append(np.nan)
                    
            else:
                ranks.append(np.nan)
                out_of_ranks.append(np.nan)
                years.append(np.nan)
                
        else:
            ranks.append(np.nan)
            out_of_ranks.append(np.nan)
            years.append(np.nan)



    

    for row in rows:
        col = row.find("td", class_="col-placement")
        
        if col:
            text = col.text.strip()
            
            if "Average Package" in text:
                # 1. Split to get the left side
                raw_avg = text.split("Average Package")[0]
                
                # 2. Remove '₹' and use .strip() to remove the '\xa0' and extra spaces
                clean_avg = raw_avg.replace('₹', '').strip()
                
                avg_packages.append(clean_avg)
            else:
                avg_packages.append(np.nan)
                
        else:
            avg_packages.append(np.nan)




    
    for row in rows:
        col = row.find("td", class_="col-placement")
        
        if col:
            text = col.text.strip()
            
            if "Highest Package" in text:
                parts = text.split("Highest Package")[0].strip()
                raw_high = parts.split("₹")[-1]
                clean_high = raw_high.replace('₹', '').strip()
                high_packages.append(clean_high)
            else:
                high_packages.append(np.nan)
                
        else:
            high_packages.append(np.nan)




    app=soup.find_all('div',class_='jsx-3698117056 mt-1 font-weight-medium text-sm')
    for i in app:
        # print(i.text)
        if 'Approved' in i.text:
            approves.append(i.text.split('|')[1].rstrip('Approved'))
        else:
            approves.append(np.nan)
    

In [32]:
len(names)

310

In [33]:
df_bcom = pd.DataFrame({"College Names":names,
                      "Degress":"B.Com",
                      "Course":courses,
                      "City":cities,
                      "State":states,
                      "Rating for 5":ratings,
                      "Rank":ranks,
                      "Total Rank":out_of_ranks,
                      "Year of Rank":years,
                      "Fees in Ruppes":fees,
                      "Average Package":avg_packages,
                      "Highest Package":high_packages,
                      "Approved":approves})

In [34]:
df_bcom

,College Names,Degress,Course,City,State,Rating for 5,Rank,Total Rank,Year of Rank,Fees in Ruppes,Average Package,Highest Package,Approved
0,Shri Ram College of Commerce,B.Com,B.Com {Hons.},New Delhi,Delhi NCR,4.3,1,317,2025,"₹ 1,25,080","9,90,000","36,00,000",AICTE
1,Hindu College,B.Com,B.Com {Hons.},New Delhi,Delhi NCR,4.2,2,317,2025,"₹ 86,010","8,40,000","36,50,000",UGC
2,Lady Shri Ram College for Women,B.Com,B.Com {Hons.},New Delhi,Delhi NCR,4.3,3,317,2025,"₹ 71,260","12,18,000","45,00,000","UGC, MHRD"
3,Hansraj College,B.Com,B.Com {Hons.},New Delhi,Delhi NCR,4.2,4,317,2025,"₹ 91,715","9,00,000","23,00,000",UGC
4,Loyola College,B.Com,B.Com,Chennai,Tamil Nadu,4.3,5,317,2025,"₹ 95,400","6,34,000","22,50,000",UGC
...,...,...,...,...,...,...,...,...,...,...,...,...,...
305,Arka Jain University,B.Com,B.Com {Hons.},Jamshedpur,Jharkhand,4,306,317,2025,"₹ 1,92,000","3,20,000","23,00,000","AICTE, PCI, INC, BCI, UGC"
306,St. Pauls College,B.Com,B.Com,Bangalore,Karnataka,3.7,307,317,2025,"₹ 1,38,000",NaN,NaN,NaN
307,SS Jain Subodh PG Mahila Mahavidyalaya,B.Com,B.Com Business Administration,Jaipur,Rajasthan,3.9,308,317,2025,"₹ 56,550",NaN,NaN,NCTE
308,J.P. College of Arts and Science,B.Com,B.Com Computer Applications,Tenkasi,Tamil Nadu,4.1,309,317,2025,"₹ 8,925",NaN,NaN,AICTE


In [35]:
df_bcom.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 310 entries, 0 to 309
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   College Names    310 non-null    object
 1   Degress          310 non-null    object
 2   Course           310 non-null    object
 3   City             310 non-null    object
 4   State            310 non-null    object
 5   Rating for 5     303 non-null    object
 6   Rank             310 non-null    object
 7   Total Rank       310 non-null    object
 8   Year of Rank     310 non-null    object
 9   Fees in Ruppes   307 non-null    object
 10  Average Package  136 non-null    object
 11  Highest Package  184 non-null    object
 12  Approved         193 non-null    object
dtypes: object(13)
memory usage: 31.6+ KB


In [36]:
courses = []
names = []
cities = []
states = []
fees = []
ratings = []
ranks = []
out_of_ranks = []
years = []
avg_packages = []
high_packages = []
approves = []

for page_no in range(25):  
    url = f"https://collegedunia.com/bsc-colleges?page={page_no}"
    page = requests.get(url, headers=headers)
    soup = BeautifulSoup(page.text)

    

    co = soup.find_all("span", class_="jsx-3698117056 fee-shorm-form")
    for i in co:
        courses.append(i.text.strip())

        
        
    na = soup.find_all("h3", class_="jsx-3698117056 font-weight-medium text-lg mb-0")
    for i in na:
        name = i.text.strip()
        name = re.sub(r",.*|\[.*?\]|-", "", name).strip()
        names.append(name)

        

    ci = soup.find_all("span",class_ = "jsx-3698117056 pr-1 location")
    for i in ci:
        city = i.text.split(",")[0]
        cities.append(city)

        

    for i in ci:
        state = i.text.split(",")[1]
        states.append(state)


        

    pr = soup.find_all("span",class_ = "jsx-3698117056 text-lg text-green d-block font-weight-bold mb-1")
    for i in pr:
        price = i.text
        if price == "--":
            fees.append(np.nan)
        else:
            fees.append(price)

    


    
    rows = soup.find_all("tr", class_="table-row") 
    
    for row in rows:
        review_col = row.find("td", class_="col-reviews")
        
        if review_col:
            valid_rating = review_col.find("span", class_="lr-key")
            
            if valid_rating:
                raw_text = valid_rating.text.strip()
                clean_rating = raw_text.split('/')[0].strip()
                ratings.append(clean_rating)
            else:
                ratings.append(np.nan)
                
        else:
            ratings.append(np.nan)





    
    for row in rows:
        tag = row.find("span", class_="jsx-2794970405")
        
        if tag:
            text = tag.text.strip()
            
            if "#" in text and "/" in text:
                ranks.append(text.split("#")[1].split("th")[0].split("/")[0])
                out_of_ranks.append(text.split("/")[1].split()[0])
                
                last_word = text.split()[-1]
                if last_word.isdigit() and len(last_word) == 4:
                    years.append(last_word)
                else:
                    years.append(np.nan)
                    
            else:
                ranks.append(np.nan)
                out_of_ranks.append(np.nan)
                years.append(np.nan)
                
        else:
            ranks.append(np.nan)
            out_of_ranks.append(np.nan)
            years.append(np.nan)



    

    for row in rows:
        col = row.find("td", class_="col-placement")
        
        if col:
            text = col.text.strip()
            
            if "Average Package" in text:
                # 1. Split to get the left side
                raw_avg = text.split("Average Package")[0]
                
                # 2. Remove '₹' and use .strip() to remove the '\xa0' and extra spaces
                clean_avg = raw_avg.replace('₹', '').strip()
                
                avg_packages.append(clean_avg)
            else:
                avg_packages.append(np.nan)
                
        else:
            avg_packages.append(np.nan)




    
    for row in rows:
        col = row.find("td", class_="col-placement")
        
        if col:
            text = col.text.strip()
            
            if "Highest Package" in text:
                parts = text.split("Highest Package")[0].strip()
                raw_high = parts.split("₹")[-1]
                clean_high = raw_high.replace('₹', '').strip()
                high_packages.append(clean_high)
            else:
                high_packages.append(np.nan)
                
        else:
            high_packages.append(np.nan)




    app=soup.find_all('div',class_='jsx-3698117056 mt-1 font-weight-medium text-sm')
    for i in app:
        # print(i.text)
        if 'Approved' in i.text:
            approves.append(i.text.split('|')[1].rstrip('Approved'))
        else:
            approves.append(np.nan)
    

In [37]:
len(names)

270

In [38]:
df_bsc = pd.DataFrame({"College Names":names,
                      "Degress":"B.Sc",
                      "Course":courses,
                      "City":cities,
                      "State":states,
                      "Rating for 5":ratings,
                      "Rank":ranks,
                      "Total Rank":out_of_ranks,
                      "Year of Rank":years,
                      "Fees in Ruppes":fees,
                      "Average Package":avg_packages,
                      "Highest Package":high_packages,
                      "Approved":approves})

In [39]:
df_bsc

,College Names,Degress,Course,City,State,Rating for 5,Rank,Total Rank,Year of Rank,Fees in Ruppes,Average Package,Highest Package,Approved
0,Hindu College,B.Sc,B.Sc {Hons.} Statistics,New Delhi,Delhi NCR,4.2,1,277,2025,"₹ 88,260","8,40,000","36,50,000",UGC
1,St Stephen's College,B.Sc,B.Sc {Hons.} Mathematics,New Delhi,Delhi NCR,4.3,2,277,2025,"₹ 88,869","9,30,000","20,00,000",UGC
2,Lady Shri Ram College for Women,B.Sc,B.Sc {Hons.} Statistics,New Delhi,Delhi NCR,4.3,3,277,2025,"₹ 71,560","12,18,000","45,00,000","UGC, MHRD"
3,Kirori Mal College,B.Sc,B.Sc Applied Physical Science with Analytical ...,New Delhi,Delhi NCR,4.2,4,277,2025,"₹ 51,345","10,80,000","24,80,000",NaN
4,St. Xavier's College,B.Sc,B.Sc {Hons.} Microbiology,Kolkata,West Bengal,4.1,5,277,2025,"₹ 4,81,700",NaN,"36,00,000",UGC
...,...,...,...,...,...,...,...,...,...,...,...,...,...
265,Soundarya Institute of Management and Science,B.Sc,B.Sc Forensic Sciences,Bangalore,Karnataka,4,270,277,2025,"₹ 3,25,000","6,12,000","12,50,000","AICTE, UGC"
266,Shri. Chetan Manju Desai College,B.Sc,B.Sc Chemistry,Cancona,Goa,NaN,271,277,2025,"₹ 25,855",NaN,NaN,NaN
267,Sri GCSR College of Education,B.Sc,B.Maths {Hons.},Srikakulam,Andhra Pradesh,4.1,272,277,2025,"₹ 48,000","1,81,000","3,50,000",NCTE
268,Government College for Women,B.Sc,B.Sc Computer Science,Mandya,Karnataka,NaN,273,277,2025,NaN,NaN,NaN,NaN


In [40]:
df_bsc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270 entries, 0 to 269
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   College Names    270 non-null    object
 1   Degress          270 non-null    object
 2   Course           270 non-null    object
 3   City             270 non-null    object
 4   State            270 non-null    object
 5   Rating for 5     263 non-null    object
 6   Rank             270 non-null    object
 7   Total Rank       270 non-null    object
 8   Year of Rank     270 non-null    object
 9   Fees in Ruppes   264 non-null    object
 10  Average Package  115 non-null    object
 11  Highest Package  153 non-null    object
 12  Approved         176 non-null    object
dtypes: object(13)
memory usage: 27.6+ KB


In [41]:
courses = []
names = []
cities = []
states = []
fees = []
ratings = []
ranks = []
out_of_ranks = []
years = []
avg_packages = []
high_packages = []
approves = []

for page_no in range(23):  
    url = f"https://collegedunia.com/ba-colleges?page={page_no}"
    page = requests.get(url, headers=headers)
    soup = BeautifulSoup(page.text)

    

    co = soup.find_all("span", class_="jsx-3698117056 fee-shorm-form")
    for i in co:
        courses.append(i.text.strip())

        
        
    na = soup.find_all("h3", class_="jsx-3698117056 font-weight-medium text-lg mb-0")
    for i in na:
        name = i.text.strip()
        name = re.sub(r",.*|\[.*?\]|-", "", name).strip()
        names.append(name)

        

    ci = soup.find_all("span",class_ = "jsx-3698117056 pr-1 location")
    for i in ci:
        city = i.text.split(",")[0]
        cities.append(city)

        

    for i in ci:
        state = i.text.split(",")[1]
        states.append(state)


        

    pr = soup.find_all("span",class_ = "jsx-3698117056 text-lg text-green d-block font-weight-bold mb-1")
    for i in pr:
        price = i.text
        if price == "--":
            fees.append(np.nan)
        else:
            fees.append(price)

    


    
    rows = soup.find_all("tr", class_="table-row") 
    
    for row in rows:
        review_col = row.find("td", class_="col-reviews")
        
        if review_col:
            valid_rating = review_col.find("span", class_="lr-key")
            
            if valid_rating:
                raw_text = valid_rating.text.strip()
                clean_rating = raw_text.split('/')[0].strip()
                ratings.append(clean_rating)
            else:
                ratings.append(np.nan)
                
        else:
            ratings.append(np.nan)





    
    for row in rows:
        tag = row.find("span", class_="jsx-2794970405")
        
        if tag:
            text = tag.text.strip()
            
            if "#" in text and "/" in text:
                ranks.append(text.split("#")[1].split("th")[0].split("/")[0])
                out_of_ranks.append(text.split("/")[1].split()[0])
                
                last_word = text.split()[-1]
                if last_word.isdigit() and len(last_word) == 4:
                    years.append(last_word)
                else:
                    years.append(np.nan)
                    
            else:
                ranks.append(np.nan)
                out_of_ranks.append(np.nan)
                years.append(np.nan)
                
        else:
            ranks.append(np.nan)
            out_of_ranks.append(np.nan)
            years.append(np.nan)



    

    for row in rows:
        col = row.find("td", class_="col-placement")
        
        if col:
            text = col.text.strip()
            
            if "Average Package" in text:
                # 1. Split to get the left side
                raw_avg = text.split("Average Package")[0]
                
                # 2. Remove '₹' and use .strip() to remove the '\xa0' and extra spaces
                clean_avg = raw_avg.replace('₹', '').strip()
                
                avg_packages.append(clean_avg)
            else:
                avg_packages.append(np.nan)
                
        else:
            avg_packages.append(np.nan)




    
    for row in rows:
        col = row.find("td", class_="col-placement")
        
        if col:
            text = col.text.strip()
            
            if "Highest Package" in text:
                parts = text.split("Highest Package")[0].strip()
                raw_high = parts.split("₹")[-1]
                clean_high = raw_high.replace('₹', '').strip()
                high_packages.append(clean_high)
            else:
                high_packages.append(np.nan)
                
        else:
            high_packages.append(np.nan)




    app=soup.find_all('div',class_='jsx-3698117056 mt-1 font-weight-medium text-sm')
    for i in app:
        # print(i.text)
        if 'Approved' in i.text:
            approves.append(i.text.split('|')[1].rstrip('Approved'))
        else:
            approves.append(np.nan)
    

In [42]:
len(names)

250

In [43]:
df_ba = pd.DataFrame({"College Names":names,
                      "Degress":"BA",
                      "Course":courses,
                      "City":cities,
                      "State":states,
                      "Rating for 5":ratings,
                      "Rank":ranks,
                      "Total Rank":out_of_ranks,
                      "Year of Rank":years,
                      "Fees in Ruppes":fees,
                      "Average Package":avg_packages,
                      "Highest Package":high_packages,
                      "Approved":approves})

In [44]:
df_ba

,College Names,Degress,Course,City,State,Rating for 5,Rank,Total Rank,Year of Rank,Fees in Ruppes,Average Package,Highest Package,Approved
0,Hindu College,BA,BA {Hons.} Political Science,New Delhi,Delhi NCR,4.2,1,255,2025,"₹ 86,010","8,40,000","36,50,000",UGC
1,St Stephen's College,BA,BA,New Delhi,Delhi NCR,4.3,2,255,2025,"₹ 88,311","9,30,000","20,00,000",UGC
2,St. Xavier's College,BA,BA {Hons.} English,Kolkata,West Bengal,4.1,3,255,2025,"₹ 1,92,900",NaN,"36,00,000",UGC
3,Lady Shri Ram College for Women,BA,BA {Hons.} Psychology,New Delhi,Delhi NCR,4.3,4,255,2025,"₹ 60,460","12,18,000","45,00,000","UGC, MHRD"
4,Miranda House,BA,BA {Hons.} Political Science,New Delhi,Delhi NCR,4.2,5,255,2025,"₹ 48,180","7,60,000","19,00,000",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,Nazareth College of Arts & Science,BA,BA English,Avadi,Tamil Nadu,2.9,243,255,2025,"₹ 51,000",NaN,NaN,NaN
246,Sardar Patel Mahavidyalaya,BA,BA,Chandrapur,Maharashtra,2.9,244,255,2025,"₹ 32,560",NaN,NaN,AICTE
247,Thiruthangal Nadar College,BA,BA Criminology and Police Administration,Chennai,Tamil Nadu,3.5,245,255,2025,"₹ 73,500",NaN,NaN,NaN
248,K.S.R. College of Arts and Science for Women,BA,BA English,Tiruchengodu,Tamil Nadu,4.2,246,255,2025,"₹ 19,080",NaN,NaN,NaN


In [45]:
df_ba.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   College Names    250 non-null    object
 1   Degress          250 non-null    object
 2   Course           250 non-null    object
 3   City             250 non-null    object
 4   State            250 non-null    object
 5   Rating for 5     241 non-null    object
 6   Rank             250 non-null    object
 7   Total Rank       250 non-null    object
 8   Year of Rank     250 non-null    object
 9   Fees in Ruppes   249 non-null    object
 10  Average Package  96 non-null     object
 11  Highest Package  131 non-null    object
 12  Approved         148 non-null    object
dtypes: object(13)
memory usage: 25.5+ KB


In [46]:
courses = []
names = []
cities = []
states = []
fees = []
ratings = []
ranks = []
out_of_ranks = []
years = []
avg_packages = []
high_packages = []
approves = []

for page_no in range(23):  
    url = f"https://collegedunia.com/bca-colleges?page={page_no}"
    page = requests.get(url, headers=headers)
    soup = BeautifulSoup(page.text)

    

    co = soup.find_all("span", class_="jsx-3698117056 fee-shorm-form")
    for i in co:
        courses.append(i.text.strip())

        
        
    na = soup.find_all("h3", class_="jsx-3698117056 font-weight-medium text-lg mb-0")
    for i in na:
        name = i.text.strip()
        name = re.sub(r",.*|\[.*?\]|-", "", name).strip()
        names.append(name)

        

    ci = soup.find_all("span",class_ = "jsx-3698117056 pr-1 location")
    for i in ci:
        city = i.text.split(",")[0]
        cities.append(city)

        

    for i in ci:
        state = i.text.split(",")[1]
        states.append(state)


        

    pr = soup.find_all("span",class_ = "jsx-3698117056 text-lg text-green d-block font-weight-bold mb-1")
    for i in pr:
        price = i.text
        if price == "--":
            fees.append(np.nan)
        else:
            fees.append(price)

    


    
    rows = soup.find_all("tr", class_="table-row") 
    
    for row in rows:
        review_col = row.find("td", class_="col-reviews")
        
        if review_col:
            valid_rating = review_col.find("span", class_="lr-key")
            
            if valid_rating:
                raw_text = valid_rating.text.strip()
                clean_rating = raw_text.split('/')[0].strip()
                ratings.append(clean_rating)
            else:
                ratings.append(np.nan)
                
        else:
            ratings.append(np.nan)





    
    for row in rows:
        tag = row.find("span", class_="jsx-2794970405")
        
        if tag:
            text = tag.text.strip()
            
            if "#" in text and "/" in text:
                ranks.append(text.split("#")[1].split("th")[0].split("/")[0])
                out_of_ranks.append(text.split("/")[1].split()[0])
                
                last_word = text.split()[-1]
                if last_word.isdigit() and len(last_word) == 4:
                    years.append(last_word)
                else:
                    years.append(np.nan)
                    
            else:
                ranks.append(np.nan)
                out_of_ranks.append(np.nan)
                years.append(np.nan)
                
        else:
            ranks.append(np.nan)
            out_of_ranks.append(np.nan)
            years.append(np.nan)



    

    for row in rows:
        col = row.find("td", class_="col-placement")
        
        if col:
            text = col.text.strip()
            
            if "Average Package" in text:
                # 1. Split to get the left side
                raw_avg = text.split("Average Package")[0]
                
                # 2. Remove '₹' and use .strip() to remove the '\xa0' and extra spaces
                clean_avg = raw_avg.replace('₹', '').strip()
                
                avg_packages.append(clean_avg)
            else:
                avg_packages.append(np.nan)
                
        else:
            avg_packages.append(np.nan)




    
    for row in rows:
        col = row.find("td", class_="col-placement")
        
        if col:
            text = col.text.strip()
            
            if "Highest Package" in text:
                parts = text.split("Highest Package")[0].strip()
                raw_high = parts.split("₹")[-1]
                clean_high = raw_high.replace('₹', '').strip()
                high_packages.append(clean_high)
            else:
                high_packages.append(np.nan)
                
        else:
            high_packages.append(np.nan)




    app=soup.find_all('div',class_='jsx-3698117056 mt-1 font-weight-medium text-sm')
    for i in app:
        # print(i.text)
        if 'Approved' in i.text:
            approves.append(i.text.split('|')[1].rstrip('Approved'))
        else:
            approves.append(np.nan)
    

In [47]:
len(names)

250

In [48]:
df_bca = pd.DataFrame({"College Names":names,
                      "Degress":"BCA",
                      "Course":courses,
                      "City":cities,
                      "State":states,
                      "Rating for 5":ratings,
                      "Rank":ranks,
                      "Total Rank":out_of_ranks,
                      "Year of Rank":years,
                      "Fees in Ruppes":fees,
                      "Average Package":avg_packages,
                      "Highest Package":high_packages,
                      "Approved":approves})

In [49]:
df_bca

,College Names,Degress,Course,City,State,Rating for 5,Rank,Total Rank,Year of Rank,Fees in Ruppes,Average Package,Highest Package,Approved
0,Christ University,BCA,BCA,Bangalore,Karnataka,4.1,1,251,2025,"₹ 5,90,000","7,00,000","32,00,000","AICTE, UGC, NBA"
1,Kristu Jayanti University,BCA,BCA,Bangalore,Karnataka,4.1,3,251,2025,"₹ 3,80,000",NaN,NaN,"AICTE, BCI, UGC, ACCA"
2,Loyola College,BCA,BCA,Chennai,Tamil Nadu,4.3,4,251,2025,"₹ 3,29,100","6,34,000","22,50,000",UGC
3,St Joseph's University,BCA,BCA,Bangalore,Karnataka,4.1,5,251,2025,"₹ 5,30,000",NaN,NaN,"UGC, MHRD"
4,Maharaja Surajmal Institute,BCA,BCA,New Delhi,Delhi NCR,3.8,6,251,2025,"₹ 3,42,900","4,50,000","12,00,000","NCTE, AICTE, BCI, UGC, QCI"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,Dr Ambedkar College,BCA,BCA,Nagpur,Maharashtra,3.9,NaN,NaN,NaN,"₹ 72,000",NaN,NaN,NaN
246,Sree Narayana Guru College,BCA,BCA,Coimbatore,Tamil Nadu,3.7,128,193,2025,"₹ 63,000",NaN,NaN,UGC
247,Kongu Engineering College,BCA,B.Sc Software System,Erode,Tamil Nadu,4.1,NaN,NaN,NaN,NaN,"5,50,000","24,50,000","AICTE, NBA"
248,Roorkee Institute of Technology,BCA,BCA,Roorkee,Uttarakhand,4.3,NaN,NaN,NaN,"₹ 1,95,000","8,00,000","18,00,000",AICTE


In [50]:
df_bca.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   College Names    250 non-null    object
 1   Degress          250 non-null    object
 2   Course           250 non-null    object
 3   City             250 non-null    object
 4   State            250 non-null    object
 5   Rating for 5     247 non-null    object
 6   Rank             244 non-null    object
 7   Total Rank       244 non-null    object
 8   Year of Rank     244 non-null    object
 9   Fees in Ruppes   247 non-null    object
 10  Average Package  121 non-null    object
 11  Highest Package  161 non-null    object
 12  Approved         190 non-null    object
dtypes: object(13)
memory usage: 25.5+ KB


In [51]:
df_bca["Degress"] = "BCA"

In [52]:
df_bca.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   College Names    250 non-null    object
 1   Degress          250 non-null    object
 2   Course           250 non-null    object
 3   City             250 non-null    object
 4   State            250 non-null    object
 5   Rating for 5     247 non-null    object
 6   Rank             244 non-null    object
 7   Total Rank       244 non-null    object
 8   Year of Rank     244 non-null    object
 9   Fees in Ruppes   247 non-null    object
 10  Average Package  121 non-null    object
 11  Highest Package  161 non-null    object
 12  Approved         190 non-null    object
dtypes: object(13)
memory usage: 25.5+ KB


In [53]:
len(df_btech)

420

In [61]:
df_btech.to_csv("./Data sets/B.Tech data.csv")

In [62]:
df_mtech.to_csv("./Data sets/M.Tech data.csv")

In [63]:
df_mba.to_csv("./Data sets/MBA data.csv")

In [64]:
df_bsc.to_csv("./Data sets/BSC data.csv")

In [65]:
df_bcom.to_csv("./Data sets/B.Com data.csv")

In [66]:
df_ba.to_csv("./Data sets/BA data.csv")

In [67]:
df_bca.to_csv("./Data sets/BCA data.csv")